# Visualização Geoespacial com Folium

Atividade prática de mapeamento de imóveis em Nova Iguaçu e Queimados.

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

## 1. Gerando a base sintética

In [ ]:
np.random.seed(42)
n_imoveis = 45

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)

centro_mapa = [df_mapa['latitude'].mean(), df_mapa['longitude'].mean()]
df_mapa.head()

## Parte 1 — Marcadores básicos

In [ ]:
mapa_basico = folium.Map(location=centro_mapa, zoom_start=12, tiles='OpenStreetMap')

for _, linha in df_mapa.head(5).iterrows():
    popup = f"{linha['tipo']} - R$ {linha['valor_venda']:,.2f}"
    folium.Marker(
        location=[linha['latitude'], linha['longitude']],
        popup=popup
    ).add_to(mapa_basico)

mapa_basico

## Parte 2 — CircleMarker

In [ ]:
mapa_circular = folium.Map(location=centro_mapa, zoom_start=12, tiles='OpenStreetMap')

cores_cidade = {
    'Nova Iguaçu': 'blue',
    'Queimados': 'orange'
}

for _, linha in df_mapa.iterrows():
    cor = cores_cidade[linha['cidade']]
    folium.CircleMarker(
        location=[linha['latitude'], linha['longitude']],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.75,
        tooltip='Clique para detalhes',
        popup=f"{linha['tipo']} - R$ {linha['valor_venda']:,.2f}"
    ).add_to(mapa_circular)

mapa_circular

## Parte 3 — MarkerCluster

In [ ]:
mapa_cluster = folium.Map(location=centro_mapa, zoom_start=12, tiles='OpenStreetMap')
cluster = MarkerCluster().add_to(mapa_cluster)

cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

for _, linha in df_mapa.iterrows():
    folium.Marker(
        location=[linha['latitude'], linha['longitude']],
        popup=f"{linha['tipo']} - R$ {linha['valor_venda']:,.2f}",
        icon=folium.Icon(color=cores_tipo[linha['tipo']], icon='home', prefix='fa')
    ).add_to(cluster)

mapa_cluster.save('mapa_imoveis_baixada.html')
mapa_cluster